# Bridge Risk Intelligence Prototype

Notebook này là phiên bản presentation của project. Code chính nằm trong `src/bridge_risk_pipeline.py` và đã được comment bằng tiếng Việt để dễ học và giải thích.

Bối cảnh: Career Experience Practicum với IHI, tự động hóa bước sàng lọc dữ liệu kiểm định cầu và tạo danh sách ưu tiên cho kỹ sư review.


## 1. Ý tưởng bài toán

Ta dùng dữ liệu cầu năm hiện tại để dự đoán năm sau rating có giảm hay không. Đây là bài toán hỗ trợ ưu tiên kiểm tra, không phải công cụ chứng nhận an toàn cầu.


> **📝 Note ôn tập — Bài toán dự đoán**
>
> - 🎯 **Mục tiêu:** Xác định xem 1 cây cầu có khả năng bị GIẢM rating (xuống cấp) vào năm sau hay không, để biết cầu nào cần kỹ sư ưu tiên đi kiểm tra trước.
> - 📥 **Input:** Dữ liệu kiểm định cầu của năm hiện tại (rating, tuổi cầu, vật liệu, lưu lượng xe, v.v.).
> - 📤 **Output:** Một dự đoán dạng xác suất/nhãn: cầu này có nguy cơ xuống cấp hay không ở lần kiểm định tiếp theo.


> **📝 Note ôn tập — Thiết lập môi trường (setup)**
>
> - 🎯 **Mục tiêu:** Xác định đúng thư mục gốc của project và thêm vào sys.path, để có thể import được module 'src.bridge_risk_pipeline' dù chạy notebook từ thư mục notebooks/ hay từ root.
> - 📥 **Input:** Không cần input dữ liệu, chỉ cần biết notebook đang được chạy từ đâu (Path.cwd()).
> - 📤 **Output:** Biến PROJECT_ROOT (đường dẫn gốc project) đã sẵn sàng để dùng ở các bước sau.


In [ ]:
# Import Path để xử lý đường dẫn project.
from pathlib import Path

# Import sys để thêm project root vào Python path.
import sys

# Import pandas để đọc file output dạng bảng.
import pandas as pd

# Import plotly để vẽ chart nhanh trong notebook.
import plotly.express as px

# Lấy thư mục gốc của project.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Thêm project root vào sys.path để import được src.
sys.path.insert(0, str(PROJECT_ROOT))

# In ra đường dẫn để kiểm tra notebook đang trỏ đúng project.
print(PROJECT_ROOT)


## 2. Chạy pipeline

Cell dưới sẽ train model và tạo lại các file trong `outputs/`, `reports/`, và `models/`. Nếu chỉ muốn xem kết quả có sẵn, có thể bỏ qua cell này.


> **📝 Note ôn tập — Chạy pipeline (train model)**
>
> - 🎯 **Mục tiêu:** Chạy toàn bộ quy trình xử lý dữ liệu + train model 1 lần bằng 1 hàm duy nhất (để không phải chạy tay từng bước, dễ tái sử dụng và dễ giải thích khi present).
> - 📥 **Input:** Dữ liệu thô của project (nằm trong thư mục data/ của project, được hàm run_pipeline tự đọc vào).
> - 📤 **Output:** Các file kết quả được tự động tạo/ghi đè trong outputs/ (danh sách ưu tiên), reports/ (thống kê), và models/ (model đã train).


In [ ]:
# Import hàm run_pipeline từ source code chính.
from src.bridge_risk_pipeline import run_pipeline

# Chạy pipeline ở chế độ fast để demo nhanh.
# fast=True dùng Decision Tree nhỏ hơn, phù hợp khi present.
results = run_pipeline(PROJECT_ROOT, fast=True)


## 3. Đọc kết quả đã tạo


> **📝 Note ôn tập — Đọc priority list**
>
> - 🎯 **Mục tiêu:** Xem thử pipeline vừa chạy có tạo ra kết quả hợp lý không, và biết cầu nào đang được xếp hạng rủi ro cao nhất để ưu tiên xem trước.
> - 📥 **Input:** File csv 'california_bridge_2026_priority_list.csv' được tạo ra ở bước 2 (nằm trong outputs/).
> - 📤 **Output:** Bảng dữ liệu (DataFrame) hiển thị 10 cầu có predicted risk score cao nhất.


In [ ]:
# Đường dẫn file priority list.
priority_path = PROJECT_ROOT / 'outputs' / 'california_bridge_2026_priority_list.csv'

# Đọc danh sách cầu đã được xếp hạng rủi ro.
priority = pd.read_csv(priority_path)

# Xem 10 cầu có predicted risk score cao nhất.
priority.head(10)


## 4. Xem tỷ lệ xuống cấp theo từng transition


> **📝 Note ôn tập — Tỷ lệ xuống cấp theo transition**
>
> - 🎯 **Mục tiêu:** Kiểm tra xem dữ liệu target (nhãn xuống cấp) có bị lệch (imbalance) quá nhiều giữa các giai đoạn năm hay không, để hiểu rõ hơn về bản chất dữ liệu mình đang train model.
> - 📥 **Input:** File 'target_summary.csv' trong reports/, chứa tỷ lệ xuống cấp đã tính theo từng transition.
> - 📤 **Output:** Biểu đồ cột (bar chart) thể hiện deterioration_rate theo từng transition.


In [ ]:
# Đọc file thống kê target.
target_summary = pd.read_csv(PROJECT_ROOT / 'reports' / 'target_summary.csv')

# Vẽ tỷ lệ cầu có rating giảm theo từng giai đoạn năm.
fig = px.bar(target_summary, x='transition', y='deterioration_rate', title='Observed deterioration rate by transition')

# Hiển thị chart.
fig.show()


## 5. Xem bản đồ top priority bridges


> **📝 Note ôn tập — Bản đồ top priority bridges**
>
> - 🎯 **Mục tiêu:** Trực quan hóa vị trí địa lý của các cầu rủi ro cao, giúp kỹ sư dễ hình dung khu vực nào cần chú ý (thay vì chỉ nhìn 1 bảng số liệu khô khan).
> - 📥 **Input:** 300 dòng đầu của priority list (đã lọc bỏ dòng thiếu latitude/longitude).
> - 📤 **Output:** Bản đồ tương tác (scatter map), màu sắc thể hiện predicted_deterioration_probability, hover để xem chi tiết từng cầu.


In [ ]:
# Lấy top 300 cầu ưu tiên để map không quá nặng.
map_data = priority.dropna(subset=['latitude', 'longitude']).head(300)

# Vẽ map. Màu thể hiện predicted deterioration probability.
fig = px.scatter_map(
    map_data,
    lat='latitude',
    lon='longitude',
    color='predicted_deterioration_probability',
    hover_name='bridge_id',
    hover_data=['FACILITY_CARRIED_007', 'LOCATION_009', 'LOWEST_RATING', 'BRIDGE_CONDITION', 'priority_rank'],
    zoom=4.5,
    height=600,
    title='Top 300 bridge priorities'
)

# Hiển thị map.
fig.show()


## 6. Kết luận để nói khi present

Prototype này cho thấy cách biến dữ liệu kiểm định cầu hằng năm thành một workflow tự động: đọc dữ liệu, tạo target, train model, đánh giá theo thời gian, và tạo priority ranking. Giá trị chính không phải là thay thế kỹ sư, mà là giúp kỹ sư bắt đầu từ nhóm cầu có dấu hiệu rủi ro cao hơn.
